# NHL Data Ingestion

Pulls live data from the NHLE API (`api-web.nhle.com/v1`) and loads it into the `nhl_data` MySQL database.

**Tables populated (in order):**
1. `teams` — seeded from the standings endpoint (all 32 franchises)
2. `standings` — today's snapshot (points / wins / losses / OTL per team)
3. `games` — today's scheduled / completed games
4. `players` — bio data for every player on every current roster
5. `rosters` — which player is on which team for which season

**Run the SQL scripts first:**
`01_create_database.sql` → `02` → `03` → `04` → `05` → `06`

**Requirement:** `pip install requests mysql-connector-python`

## 1 · Config — edit only this cell

In [ ]:
# --- database connection ---
DB_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "nhl_data"
DB_USER     = "root"      # change if you use a different MySQL user
DB_PASSWORD = ""          # fill in your MySQL root password

# --- season to load rosters for ---
SEASON = "20242025"       # format: YYYYYYYY  e.g. 20242025 = 2024-25 season

## 2 · API helpers

In [ ]:
import requests
from datetime import date

BASE_URL = "https://api-web.nhle.com/v1"

def api_get(endpoint):
    r = requests.get(f"{BASE_URL}/{endpoint}", timeout=10)
    r.raise_for_status()
    return r.json()

def get_standings():
    return api_get("standings/now")["standings"]

def get_schedule_today():
    return api_get("schedule/now")

def get_roster(team_abbrev, season=SEASON):
    return api_get(f"roster/{team_abbrev}/{season}")

def get_player(player_id):
    return api_get(f"player/{player_id}/landing")

print("API helpers ready")

## 3 · Connect to MySQL

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)
cur = conn.cursor()
print("Connected:", conn.get_server_info())

## 4 · Load teams + standings

The standings endpoint returns every team that has played this season, so it's the easiest way to seed the `teams` table in one shot.

In [ ]:
standings_data = get_standings()
today = date.today().isoformat()

# ── teams ────────────────────────────────────────────────────────────
# The standings payload has one entry per team with all the fields we need.
# teamAbbrev / teamName / conferenceAbbrev / divisionName are the real
# key names confirmed from the live API.
team_rows = [
    (
        t["teamAbbrev"]["default"],
        t.get("teamName", {}).get("default"),
        t.get("conferenceAbbrev"),
        t.get("divisionName"),
    )
    for t in standings_data
]

cur.executemany(
    """
    INSERT INTO teams (team_abbrev, team_name, conference, division)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        team_name  = VALUES(team_name),
        conference = VALUES(conference),
        division   = VALUES(division)
    """,
    team_rows,
)
conn.commit()
print(f"Upserted {len(team_rows)} teams")

# Build a local abbrev → team_id map for use in subsequent inserts
cur.execute("SELECT team_id, team_abbrev FROM teams")
team_id_map = {row[1]: row[0] for row in cur.fetchall()}

# ── standings ────────────────────────────────────────────────────────
standings_rows = [
    (
        team_id_map[t["teamAbbrev"]["default"]],
        today,
        t["points"],
        t["wins"],
        t["losses"],
        t["otLosses"],
    )
    for t in standings_data
    if t["teamAbbrev"]["default"] in team_id_map
]

cur.executemany(
    """
    INSERT INTO standings (team_id, standings_date, points, wins, losses, ot_losses)
    VALUES (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        points    = VALUES(points),
        wins      = VALUES(wins),
        losses    = VALUES(losses),
        ot_losses = VALUES(ot_losses)
    """,
    standings_rows,
)
conn.commit()
print(f"Upserted {len(standings_rows)} standings rows for {today}")

## 5 · Load today's games

In [ ]:
schedule = get_schedule_today()

# The schedule payload nests games under gameWeek[0].games
game_week = schedule.get("gameWeek", [])
games_raw = game_week[0].get("games", []) if game_week else []

game_rows = []
for g in games_raw:
    home_abbrev = g["homeTeam"]["abbrev"]
    away_abbrev = g["awayTeam"]["abbrev"]

    # Skip if we somehow got a team not in our teams table
    if home_abbrev not in team_id_map or away_abbrev not in team_id_map:
        print(f"  Skipping game {g.get('id')} — unknown team abbrev")
        continue

    game_rows.append((
        g["id"],                                  # game_pk — NHL's own ID
        g.get("gameDate", today),
        team_id_map[home_abbrev],
        team_id_map[away_abbrev],
        g["homeTeam"].get("score"),               # None if not yet started
        g["awayTeam"].get("score"),
        g.get("gameState", "FUT"),
    ))

if game_rows:
    cur.executemany(
        """
        INSERT INTO games
            (game_pk, game_date, home_team_id, away_team_id,
             home_score, away_score, game_state)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            home_score = VALUES(home_score),
            away_score = VALUES(away_score),
            game_state = VALUES(game_state)
        """,
        game_rows,
    )
    conn.commit()
    print(f"Upserted {len(game_rows)} games")
else:
    print("No games scheduled today")

## 6 · Load rosters + players

Loops through every team, fetches their roster, then fetches individual player bios. This makes ~32 roster calls + N player calls — expect it to take a minute or two.

In [ ]:
import time

all_abbrevs = list(team_id_map.keys())
seen_player_ids = set()   # avoid re-fetching the same player from multiple teams

for abbrev in all_abbrevs:
    print(f"  Loading roster: {abbrev} ... ", end="", flush=True)

    try:
        roster = get_roster(abbrev, SEASON)
    except Exception as e:
        print(f"SKIP ({e})")
        continue

    # The roster payload groups players by position group:
    # forwards / defensemen / goalies  — each is a list of player dicts
    all_players = (
        roster.get("forwards", []) +
        roster.get("defensemen", []) +
        roster.get("goalies", [])
    )

    roster_rows = []

    for p in all_players:
        pid = p["id"]

        # ── players table ────────────────────────────────────────────
        if pid not in seen_player_ids:
            seen_player_ids.add(pid)
            # firstName / lastName come as {"default": "..."} objects
            first = p.get("firstName", {}).get("default", "")
            last  = p.get("lastName",  {}).get("default", "")
            pos   = p.get("positionCode")   # e.g. 'C', 'LW', 'D', 'G'
            bdate = p.get("birthDate")      # YYYY-MM-DD string or None

            cur.execute(
                """
                INSERT INTO players (player_id, first_name, last_name, position, birth_date)
                VALUES (%s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    first_name = VALUES(first_name),
                    last_name  = VALUES(last_name),
                    position   = VALUES(position),
                    birth_date = VALUES(birth_date)
                """,
                (pid, first, last, pos, bdate),
            )

        # ── rosters table ────────────────────────────────────────────
        roster_rows.append((team_id_map[abbrev], pid, SEASON))

    if roster_rows:
        cur.executemany(
            """
            INSERT IGNORE INTO rosters (team_id, player_id, season)
            VALUES (%s, %s, %s)
            """,
            roster_rows,
        )

    conn.commit()
    print(f"{len(all_players)} players")
    time.sleep(0.2)   # be polite to the API

print(f"\nDone. Total unique players seen: {len(seen_player_ids)}")

## 7 · Verify

In [ ]:
checks = [
    ("teams",     "SELECT COUNT(*) FROM teams"),
    ("standings", "SELECT COUNT(*) FROM standings"),
    ("games",     "SELECT COUNT(*) FROM games"),
    ("players",   "SELECT COUNT(*) FROM players"),
    ("rosters",   "SELECT COUNT(*) FROM rosters"),
]

print("Row counts:")
for label, sql in checks:
    cur.execute(sql)
    n = cur.fetchone()[0]
    print(f"  {label:12} {n:>6,}")

print("\nTop 5 standings:")
cur.execute("""
    SELECT t.team_abbrev, s.points, s.wins, s.losses, s.ot_losses
    FROM standings s
    JOIN teams t USING (team_id)
    WHERE s.standings_date = CURDATE()
    ORDER BY s.points DESC
    LIMIT 5
""")
for row in cur.fetchall():
    print(f"  {row[0]:4}  {row[1]} pts  ({row[2]}-{row[3]}-{row[4]})")

In [ ]:
cur.close()
conn.close()
print("Connection closed.")